In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import T5TokenizerFast, CLIPProcessor, CLIPTokenizerFast, CLIPImageProcessorFast, T5ForConditionalGeneration


from peft import LoraConfig, get_peft_model, TaskType
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os 


In [2]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TEST_IMAGE_DIR,
                            FAISS_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CLIP_MODEL_NAME,
                            T5_MODEL_NAME,
                            VLM_CHECKPOINT_DIR,
                            T5_DECODER_LORA_CONFIG)

from Modules.FusionVLM import FusionVLM, create_default_FusionVLM, load_default_FusionVLM, save_FusionVLM
from Modules.retrieval_module import Retriever
from Modules.datasets import VLMDataset, VLMDataCollator
from Modules.utils import print_model_param_stats, add_dict
from Modules.metrics import evaluate_captioning, setup_nltk

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
CLIP_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, use_fast=True, local_files_only=True)
# CLIP_tokenizer = CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME, local_files_only=True)
collator = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)

In [5]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [6]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [ ]:
import numpy as np
from torch.utils.data import DataLoader, Subset

# fraction = 0.05
# num_train_samples = int(len(train_dataset) * fraction)
# num_test_samples = int(len(test_dataset) * fraction)
num_train_samples = 1024
num_test_samples = 128


indices = np.random.choice(len(train_dataset), num_train_samples, replace=False)
train_subset = Subset(train_dataset, indices)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

indices = np.random.choice(len(test_dataset), num_test_samples, replace=False)
test_subset = Subset(test_dataset, indices)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [7]:
model = create_default_FusionVLM().to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {num_params:,}\nText Decoder", end=' ')
# model.text_decoder.print_trainable_parameters()

c:\Users\Mahan\Documents\Projects\Retrieval-Augmented-Image-Captioning\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [8]:
from Modules.FusionVLM import apply_lora_config

In [9]:
model = FusionVLM(vision_encoder_name=CLIP_MODEL_NAME,
                    text_encoder_name=T5_MODEL_NAME,
                    T5_text_decoder_name=T5_MODEL_NAME,
                    num_fusion_blocks=4,
                    use_local_files=True
                    )


In [10]:
model = apply_lora_config(model).to(DEVICE)

In [11]:
print_model_param_stats(model)

Module                                          Total    Trainable       Frozen
--------------------------------------------------------------------------------
vision_encoder                             87,456,000            0   87,456,000
text_encoder                              109,628,544            0  109,628,544
vision_proj                                   590,592      590,592            0
text_proj                                     590,592      590,592            0
fusion_blocks                              56,724,480   56,724,480            0
post_fusion_ln                                  1,536        1,536            0
fusion_proj                                   590,592      590,592            0
text_decoder                              254,655,744   31,752,192  222,903,552
--------------------------------------------------------------------------------
TOTAL                                     510,238,080   90,249,984  419,988,096


In [ ]:
print_model_param_stats(model)

In [12]:
NUM_EPOCHS = 2
full_history = {}
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

setup_nltk()
os.makedirs(VLM_CHECKPOINT_DIR, exist_ok=True)
# scaler = GradScaler()

In [13]:
def evaluate_epoch(model, loader, tokenizer):
    model.eval()

    loss = 0
    preds = []
    refs = []

    with torch.no_grad():
        for batch in loader:
            gt_captions = batch["all_captions"]  # List[List[str]]
            
            outputs = model(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch["retrieved_pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
            )
            loss += outputs.loss.item()
            
            generated_ids = model.generate(
                    query_pixel_values=batch["query_pixel_values"],
                    retrieved_pixel_values=batch["retrieved_pixel_values"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    max_length=64,
                    # num_beams=3
            )

            decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            preds.extend(decoded)
            refs.extend(gt_captions)
            
    loss = loss / len(loader)
    return loss, evaluate_captioning(preds, refs)

In [14]:
def train_epoch(model, loader, optimizer, epoch):
    model.train()
    epoch_loss = 0.0
    progress_bar = tqdm(loader, desc=f"Epoch {epoch+1}", leave=True)

    for batch in progress_bar:
        optimizer.zero_grad()

        outputs = model(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch["retrieved_pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        # print(loss.item(), end="  ")
        progress_bar.set_postfix(loss=loss.item())
        epoch_loss += loss.item()

    epoch_loss = epoch_loss / len(loader)
    progress_bar.set_postfix(loss=epoch_loss)
    # print(f"Epoch {epoch+1} Avg Loss: {epoch_loss:.4f}")
    
    return epoch_loss

In [15]:
def train_and_evaluate_model(model, train_loader, optimizer, num_epochs, test_loader=None, tokenizer=None, save_interval=5):    
    history = {'train_loss': [],
               'test_loss': []
               }
    
    for epoch in range(num_epochs):
        epoch_loss = train_epoch(model, train_loader, optimizer, epoch)
        history['train_loss'].append(epoch_loss)

        # Evaluate
        if test_loader:
            test_loss, metrics = evaluate_epoch(model, test_loader, tokenizer)
            history['test_loss'].append(test_loss)
            add_dict(history, metrics)
            
            for k, v in metrics.items():
                print(f"{k}: {v:.4f}")

        # ---- Backup every 5 epochs ----
        if (epoch + 1) % save_interval == 0:
            save_FusionVLM(model, f'epoch{epoch+1}', VLM_CHECKPOINT_DIR)
    
    return history


In [ ]:
history = train_and_evaluate_model(model, train_loader, optimizer, NUM_EPOCHS, test_loader, T5_tokenizer)
add_dict(full_history, history)

Epoch 1: 100%|██████████| 1924/1924 [16:13<00:00,  1.98it/s, loss=2.33]


BLEU-1: 0.4330
BLEU-2: 0.2621
BLEU-3: 0.1591
BLEU-4: 0.0961
METEOR: 0.3343
ROUGE-L: 0.3282
CIDEr: 0.1285


Epoch 2:  46%|████▌     | 876/1924 [07:25<10:01,  1.74it/s, loss=2.73]

In [ ]:
# history = train_and_evaluate_model(model, train_loader, optimizer, save_interval=1000)

In [ ]:
full_history

In [ ]:
import json
with open('history.json', "w", encoding="utf-8") as f:
    json.dump(full_history, f, indent=2, ensure_ascii=False)

In [16]:
with torch.no_grad():
    for batch in train_loader:
        gt_captions = batch["all_captions"]  # List[List[str]]

        generated_ids = model.generate(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch["retrieved_pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=1,
            # do_sample=True,
            # top_p=0.9,
            # temperature=0.8,
            # repetition_penalty=1.2,
        )

        decoded = T5_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        print(gt_captions)
        print(decoded)            
        break

[['A tattoo artist concentrating as he works on creating a tattoo for a customer .', 'A tattoo artist applying tattoo ink to the skin .', 'A tattoo artist working on somebody .', 'A guy operating on human eyes .', 'A tattoo artist at work .'], ['A man with gray hair and glasses looks downwards .', 'A closeup of an elderly man wearing glasses .', 'A man wearing glasses looks down and smiles .', 'An elderly man looks down and smiles .', 'Gray-haired man with glasses .'], ['There is one football player with the ball with one of his teammates following along with a member of the other team trying to catch up .', 'Football players on a field ; there are three players  two of one team and one of their opponent  as well as a referee on the field .', 'A football player in a mostly white uniform flees as a red-uniformed member of the opposite team attempts to catch and tackle him .', 'A guy is running down the field holding a football while his teammate  an opponent  and referee run behind him 

In [ ]:
batch['attention_mask']